In [1]:
import numpy as np
import pandas as pd
import os

In [2]:
data_f = "/ix/djishnu/Aaron_F/proteomics_proj/Data/phosphoproteomics/TMT_data.csv"
sample_name_f = "/ix/djishnu/Aaron_F/proteomics_proj/Data/phosphoproteomics/TMT_columns_to_sample_names.csv"
expressed_tf_f = "/ix/djishnu/Aaron_F/Cleaned_Proteomics_Aaron/20240906/TF_enrichment/Results/log2_thresh_2.5_expressed_tfs.txt"


out_dir = "/ix/djishnu/Aaron_F/proteomics_proj/Data/phosphoproteomics"

In [3]:
# load data
df = pd.read_csv(data_f)
sample_naming = pd.read_csv(sample_name_f)
expressed_tf = np.loadtxt(expressed_tf_f, dtype=str).tolist()
print(df.head())
print(sample_naming.head())

                               id  TMT_126C_total  TMT_127N_total  \
0     Ahnak_E9Q616_S42s_1_1_42_42         84042.0         81732.0   
1     Ahnak_E9Q616_S94s_1_1_94_94       4245380.0       3940429.0   
2  Ahnak_E9Q616_S116s_1_1_116_116        544128.0        485757.0   
3  Ahnak_E9Q616_S136s_1_1_136_136        854207.0       1073467.0   
4  Ahnak_E9Q616_T159t_1_1_159_159        349598.0        419199.0   

   TMT_127C_total  TMT_128N_total  TMT_128C_total  TMT_129N_total  \
0         67567.0         88077.0        120407.0         78779.0   
1       3396104.0       4115950.0       4638805.0       3623190.0   
2        527435.0        544639.0        563640.0        552360.0   
3        801867.0        873695.0       2135641.0        823829.0   
4        288773.0        281710.0        361781.0        307972.0   

   TMT_129C_total  TMT_130N_total  TMT_130C_total  TMT_131N_total  \
0         93478.0        102299.0        102588.0         91978.0   
1       3576744.0       4712262.

In [4]:
# 1. Rename TMT columns using the sample_naming mapping
rename_map = dict(zip(sample_naming['TMT_columns'], sample_naming['sample']))
df = df.rename(columns=rename_map)

# 2. Split the id column into geneSymbol and id
df['geneSymbol'] = df['id'].str.split('_').str[0]
df['id'] = df['id'].str.split('_', n=1).str[1]

# 3. Reorder so geneSymbol and id are the first columns
sample_cols = sample_naming['sample'].tolist()
df = df[['geneSymbol', 'id'] + sample_cols]

In [5]:
has_nan = df[sample_cols].isnull().any(axis=1).sum()
no_nan = df[sample_cols].isnull().any(axis=1).sum() == 0

print(f"Rows with any NaN:     {has_nan}")
print(f"Rows without any NaN:  {len(df) - has_nan}")
print(f"Total rows:            {len(df)}")

Rows with any NaN:     604
Rows without any NaN:  21037
Total rows:            21641


#  Control Center Data

In [6]:
# --- Column definitions ---
ctrl_cols = ['PBS_0_R1', 'PBS_0_R2']
il6_cols  = {'20': ['IL6_20_R1',  'IL6_20_R2'],
             '45': ['IL6_45_R1',  'IL6_45_R2'],
             '90': ['IL6_90_R1',  'IL6_90_R2']}
il10_cols = {'20': ['IL10_20_R1', 'IL10_20_R2'],
             '45': ['IL10_45_R1', 'IL10_45_R2'],
             '90': ['IL10_90_R1', 'IL10_90_R2']}

all_sample_cols = (ctrl_cols
                   + [c for cols in il6_cols.values()  for c in cols]
                   + [c for cols in il10_cols.values() for c in cols])

# --- log2 transform ---
lfc_df = df[['id', 'geneSymbol'] + all_sample_cols].copy()
lfc_df[all_sample_cols] = np.log2(lfc_df[all_sample_cols])

# --- Subtract per-site mean of log2(control) ---
ctrl_mean = lfc_df[ctrl_cols].mean(axis=1)
for col in all_sample_cols:
    lfc_df[col] = lfc_df[col] - ctrl_mean

# --- Save ---
lfc_df.to_csv(f"{out_dir}/TMT_control_centered_log2fc.csv", index=False)
print(lfc_df.head())


                         id geneSymbol  PBS_0_R1  PBS_0_R2  IL6_20_R1  \
0     E9Q616_S42s_1_1_42_42      Ahnak -0.211800  0.211800  -0.058283   
1     E9Q616_S94s_1_1_94_94      Ahnak  0.060961 -0.060961  -0.186286   
2  E9Q616_S116s_1_1_116_116      Ahnak -0.024966  0.024966   0.011635   
3  E9Q616_S136s_1_1_136_136      Ahnak -0.035273  0.035273   1.085330   
4  E9Q616_T159t_1_1_159_159      Ahnak  0.036017 -0.036017  -0.350772   

   IL6_20_R2  IL6_45_R1  IL6_45_R2  IL6_90_R1  IL6_90_R2  IL10_20_R1  \
0  -0.081621   0.306937  -0.049927  -0.252009   0.120180   -0.526591   
1   0.029335   0.188820   0.050204  -0.046580   0.148349   -0.261052   
2   0.069790   0.025862   0.150811  -0.188677   0.082186   -0.069919   
3   1.265706   1.286738   1.356343   0.294347   0.577699   -0.126496   
4  -0.049734   0.085437  -0.270520   0.297956   0.295927   -0.239744   

   IL10_20_R2  IL10_45_R1  IL10_45_R2  IL10_90_R1  IL10_90_R2  
0   -0.138831   -0.135310   -0.116998    0.071810    0.075880  
